In [ ]:
import math
import textwrap
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
import matplotlib.patches as mpatches
from pywaffle import Waffle

# ===== Knobs =====
MM = 1 / 25.4

FIG_WIDTH_MM  = 180     # total figure width
FIG_HEIGHT_MM = 155     # total figure height (panel A + B/C row)

FONTSIZE             = 6
HEADER_FONTSIZE      = 7
PANEL_LABEL_FONTSIZE = 9   # bold "a", "b", "c" labels

# Panel A waffle config (carried over from the previous version)
WAFFLE_ROWS_A        = 5
LEGEND_LABEL_WRAP    = 22

# Panel C waffle config — vertical bars per year, this many tiles wide.
# More columns = shorter, wider bars (less figure height needed).
C_COLUMNS = 5

# ----- Top-level layout knobs (relative units) -----
# Row 0 = Panel A (spans full width).  Row 1 = Panel B (left) + C (right).
PANEL_A_H  = 1.0     # relative height of panel A row
PANEL_BC_H = 0.85    # relative height of panel B+C row
PANEL_B_W  = 0.45    # fractional width of panel B in the BC row
PANEL_C_W  = 0.50    # fractional width of panel C in the BC row

TOP_HSPACE = 0.16    # vertical gap between panel A row and the B/C row
TOP_WSPACE = 0.10    # horizontal gap between panel B and panel C

# Panel B internal padding (within its gs_top[1, 0] cell):
#   B_LEFT_PAD reserves a column on the left so the y-tick labels (which
#     are right-anchored at the spine) end up with their LEFT edge at
#     MARGIN_LEFT, aligned with panel A's row labels.
#   B_BOT_PAD reserves a row at the bottom so the rotated x-tick labels
#     and x-axis label fit within the cell (i.e. within panel C's height).
B_LEFT_PAD = 0.11
B_BOT_PAD  = 0.45

# Outer margins (fraction of figure dims). Top margin leaves room for the
# panel labels (a/b/c) to sit ABOVE the panel content. Right margin keeps
# panel A's vertical legend from clipping the figure edge.
MARGIN_LEFT   = 0.04
MARGIN_RIGHT  = 0.97
MARGIN_TOP    = 0.95
MARGIN_BOTTOM = 0.09

# Panel labels:
#   X is shared by 'a', 'b', 'd' (and 'c' uses its panel's x) so they sit on
#     a common left margin.
#   Y_OFFSET is added to each panel's top edge so labels float ~5 mm above
#     the panel content rather than overlapping its top row.
PANEL_LABEL_X = 0.005
PANEL_LABEL_Y_OFFSET = 5 / FIG_HEIGHT_MM

# Per-panel legend nudges (mm). Positive = shift up (away from cell bottom).
A_LEGEND_Y_NUDGE_MM = 4    # panel A legend stack moves up
C_LEGEND_Y_NUDGE_MM = -4   # panel C legend moves down (relative to its cell)

# Per-panel vertical shifts (mm) applied AFTER gridspec layout, to nudge
# whole panels up without disturbing the rest of the layout.
B_SHIFT_MM = 4
C_SHIFT_MM = 6

# Panel D placeholder — panel D content is composed manually in Affinity
# Designer below this label. Adjust PANEL_D_LABEL_Y to match where the
# manual panel D will sit in the final composition.
PANEL_D_LABEL_Y = 0.12 + 3 / FIG_HEIGHT_MM

# ===== Shared category metadata =====
CATEGORIES = [
    'Gene-disease Associations',
    'Variant Reclassifications',
    'Missed in Manual Analysis',
    'CNV/SV',
]
ROW_LABELS = [
    'Gene-disease\nAssociations',
    'Variant\nReclassifications',
    'Missed in\nManual Analysis',
    'CNV/SV',
]
# Per-category subcategory shades (panel A) — 11 colours grouped by category.
colors_list = [
    '#EC4E20', '#F69A7F', '#FCCCBE',              # Gene-disease (3)
    '#FF9505', '#FFC474',                          # Variant Reclass (2)
    '#016FB9', '#4799D0', '#8EC4E8', '#D4EEFF',   # Missed (4)
    '#57ABA9', '#C2FCFA',                          # CNV/SV (2)
]
COLOR_RANGES = {
    'Gene-disease Associations': (0, 3),
    'Variant Reclassifications': (3, 5),
    'Missed in Manual Analysis': (5, 9),
    'CNV/SV':                    (9, 11),
}
# Single-colour-per-category (panel C uses the "primary" of each).
CATEGORY_PRIMARY_COLOR = {
    'Gene-disease Associations': '#EC4E20',
    'Variant Reclassifications': '#FF9505',
    'Missed in Manual Analysis': '#016FB9',
    'CNV/SV':                    '#57ABA9',
}

COHORTS = [
    # (data column, header text, total diagnoses, cohort N, diagnosis %)
    ('All',     'Full\nCohort',          241,  4738, '5.1%'),
    ('NDD',     'NDD\nSub-cohort',        86,  1665, '5.2%'),
    ('Cardiac', 'Cardiac\nSub-cohort',    53,   997, '5.3%'),
    ('Renal',   'Renal\nSub-cohort',      41,   695, '5.9%'),
]

C_YEARS = [
    # (year column, total diagnoses, cohort N, yield %)
    ('2019', 73,  879, '8.3%'),
    ('2020', 68, 1068, '6.4%'),
    ('2021', 42, 1319, '3.2%'),
    ('2022', 49, 1234, '4.0%'),
]

B_GENOMES_COLOR = '#016FB9'
B_EXOMES_COLOR  = '#8EC4E8'

# ===== Data =====
# Panel A: per-cohort, per-category, with subcategory breakdowns.
data_a = pd.read_csv(
    '../../Data/Fig3/Talos_solves_VCGS-prospective_261128.csv',
    header=0, index_col=[1, 0], usecols=range(1, 7),
)
data_a_combined = data_a.groupby(level=[0, 1]).sum()

# Panel B: timeline of cumulative exome/genome counts.
data_b = pd.read_csv('../../Data/Fig3/Data_aggregation_timeline.csv')
data_b['Date'] = pd.to_datetime(data_b['Date'], format='%b-%y')

# Panel C: per-year, per-category counts (no subcategory breakdown).
data_c = pd.read_csv(
    '../../Data/Fig3/Talos_solves_by_year-prospective_261128.csv',
    header=0, index_col='Reason_Type', usecols=range(1, 7),
)
data_c = data_c[data_c.index != 'Total']
data_c_combined = data_c.groupby(level=0).sum(numeric_only=True)
# Order categories so that Gene-disease ends up at the top of the vertical
# waffle (pywaffle fills bottom-up when vertical=True).
custom_sort_c = {cat: i for i, cat in enumerate(reversed(CATEGORIES))}
data_c_sorted = data_c_combined.sort_index(key=lambda x: x.map(custom_sort_c))
# Colours in the same bottom-up order.
colors_c_reversed = [CATEGORY_PRIMARY_COLOR[cat] for cat in reversed(CATEGORIES)]

# ===== Panel A internal layout =====
A_ROW_LABEL_W = 5.0
A_LEGEND_W    = 9.0
A_HEADER_H    = 0.55
A_FOOTER_H    = 0.35
A_WAFFLE_H    = 1.0
A_HSPACE      = 0.05
A_WSPACE      = 0.10

def max_cols_for_cohort(cohort_key):
    return max(
        math.ceil(data_a_combined.loc[cat, cohort_key].sum() / WAFFLE_ROWS_A)
        for cat in CATEGORIES
    )
COHORT_WIDTHS = {c[0]: max_cols_for_cohort(c[0]) for c in COHORTS}

A_width_ratios  = ([A_ROW_LABEL_W]
                   + [COHORT_WIDTHS[c[0]] for c in COHORTS]
                   + [A_LEGEND_W])
A_height_ratios = [A_HEADER_H] + [A_WAFFLE_H] * len(CATEGORIES) + [A_FOOTER_H]

# ===== Top-level gridspec =====
gs_top = GridSpec(
    nrows=2, ncols=2,
    height_ratios=[PANEL_A_H, PANEL_BC_H],
    width_ratios=[PANEL_B_W, PANEL_C_W],
    left=MARGIN_LEFT, right=MARGIN_RIGHT,
    top=MARGIN_TOP,   bottom=MARGIN_BOTTOM,
    hspace=TOP_HSPACE, wspace=TOP_WSPACE,
)

# Panel A spans both columns of row 0.
gs_a = GridSpecFromSubplotSpec(
    nrows=len(CATEGORIES) + 2, ncols=len(COHORTS) + 2,
    subplot_spec=gs_top[0, :],
    width_ratios=A_width_ratios, height_ratios=A_height_ratios,
    hspace=A_HSPACE, wspace=A_WSPACE,
)

# ===== Build pywaffle plots dict for panel A =====
plots = {}
for col_idx, (cohort_key, *_) in enumerate(COHORTS, start=1):
    for row_idx, cat in enumerate(CATEGORIES, start=1):
        c0, c1 = COLOR_RANGES[cat]
        plots[(gs_a[row_idx, col_idx],)] = {
            'values': data_a_combined.loc[cat, cohort_key],
            'colors': colors_list[c0:c1],
        }

fig = plt.figure(
    figsize=(FIG_WIDTH_MM * MM, FIG_HEIGHT_MM * MM),
    FigureClass=Waffle,
    plots=plots,
    rows=WAFFLE_ROWS_A,
    rounding_rule='ceil',
    tight=False,
)

# ===== Panel A text labels =====
def text_axis(spec):
    ax = fig.add_subplot(spec)
    ax.axis('off')
    return ax

text_axis(gs_a[0, 0]).text(
    0.0, 0.0, 'New Diagnoses\n(Count)',
    ha='left', va='bottom', fontsize=HEADER_FONTSIZE, fontweight='bold',
)
for col_idx, (_, name, count, *_) in enumerate(COHORTS, start=1):
    text_axis(gs_a[0, col_idx]).text(
        0.5, 0.0, f"{name}\n({count})",
        ha='center', va='bottom', fontsize=HEADER_FONTSIZE,
    )
for row_idx, label in enumerate(ROW_LABELS, start=1):
    text_axis(gs_a[row_idx, 0]).text(
        0.0, 0.5, label, ha='left', va='center', fontsize=FONTSIZE,
    )
text_axis(gs_a[-1, 0]).text(
    0.0, 1.0, 'Total Cohort Size\nNew Diagnosis %',
    ha='left', va='top', fontsize=FONTSIZE, fontweight='bold',
)
for col_idx, (_, _, _, total, pct) in enumerate(COHORTS, start=1):
    text_axis(gs_a[-1, col_idx]).text(
        0.5, 1.0, f"N={total}\n{pct}",
        ha='center', va='top', fontsize=FONTSIZE,
    )

# ===== Panel A vertical legend (measure-and-stack) =====
def wrapped(text):
    return textwrap.fill(text, width=LEGEND_LABEL_WRAP)

ax_a_leg = fig.add_subplot(gs_a[1:-1, -1])
ax_a_leg.axis('off')

a_legends = []
for cat in CATEGORIES:
    c0, c1 = COLOR_RANGES[cat]
    subcats = list(data_a_combined.loc[cat].index)
    handles = [
        mpatches.Patch(color=colors_list[c0 + j], label=wrapped(s))
        for j, s in enumerate(subcats)
    ]
    leg = ax_a_leg.legend(
        handles=handles,
        loc='upper left', bbox_to_anchor=(0, 1),
        title=wrapped(cat),
        title_fontproperties={'weight': 'bold', 'size': FONTSIZE},
        fontsize=FONTSIZE, frameon=False,
        handlelength=1.2, handletextpad=0.6, borderpad=0, labelspacing=0.4,
    )
    leg._legend_box.align = "left"
    ax_a_leg.add_artist(leg)
    a_legends.append(leg)

fig.canvas.draw()
renderer = fig.canvas.get_renderer()

a_leg_pos = gs_a[1:-1, -1].get_position(fig)
# Lift the entire legend stack relative to the waffle area.
y_cursor = a_leg_pos.y1 + A_LEGEND_Y_NUDGE_MM / FIG_HEIGHT_MM
SECTION_GAP_FIG = 0.01
for leg in a_legends:
    bbox_px = leg.get_window_extent(renderer)
    height_fig = bbox_px.height / fig.bbox.height
    leg.set_bbox_to_anchor((a_leg_pos.x0, y_cursor), transform=fig.transFigure)
    y_cursor -= height_fig + SECTION_GAP_FIG

# ===== Panel B: step-line plot =====
# Nest in a 2x2 gridspec inside gs_top[1, 0]:
#   - left col (B_LEFT_PAD) reserves room for the y-tick labels so they
#     render INSIDE the figure margin, aligned with panel A row labels.
#   - bottom row (B_BOT_PAD) reserves room for the rotated x-tick labels
#     and x-axis label so the whole panel fits within panel C's height.
gs_b = GridSpecFromSubplotSpec(
    nrows=2, ncols=2,
    subplot_spec=gs_top[1, 0],
    height_ratios=[1.0, B_BOT_PAD],
    width_ratios=[B_LEFT_PAD, 1.0],
    hspace=0.0, wspace=0.0,
)
ax_b = fig.add_subplot(gs_b[0, 1])
ax_b.step(data_b['Date'], data_b['Exomes'],  label='Exomes',
          where='post', color=B_EXOMES_COLOR,  linewidth=2)
ax_b.step(data_b['Date'], data_b['Genomes'], label='Genomes',
          where='post', color=B_GENOMES_COLOR, linewidth=2)
ax_b.set_xlabel('Date of entry into reanalysis study', fontsize=FONTSIZE)
ax_b.set_ylabel('Count', fontsize=FONTSIZE)
ax_b.legend(loc='upper left', fontsize=FONTSIZE, frameon=False)
ax_b.set_xlim(data_b['Date'].min(), pd.Timestamp('2025-12-31'))
ax_b.set_ylim(0, 4000)
ax_b.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax_b.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
ax_b.tick_params(axis='x', labelsize=FONTSIZE, rotation=45)
ax_b.tick_params(axis='y', labelsize=FONTSIZE)
for tick in ax_b.get_xticklabels():
    tick.set_horizontalalignment('right')
ax_b.spines['top'].set_visible(False)
ax_b.spines['right'].set_visible(False)

# ===== Panel C: per-year vertical waffles + labels + legend =====
# Year column widths chosen so cell size matches panel A cells (uniform
# across years). With aspect='equal' and a fixed slot height, cells are
# height-constrained for the tallest year (2019, ceil(73/C_COLUMNS) rows)
# at slot_h/N_rows; narrower slot widths force the other years to also be
# height-constrained at the same cell size instead of growing wider.
N_YEARS = len(C_YEARS)
gs_c = GridSpecFromSubplotSpec(
    nrows=2, ncols=N_YEARS + 1,
    subplot_spec=gs_top[1, 1],
    height_ratios=[1.0, 0.22],          # waffle / year-label rows
    width_ratios=[1] * N_YEARS + [1.6], # equal year cols + legend col
    hspace=0.05, wspace=0.05,
)

# Track every panel-C axes so we can shift them as a group below.
panel_c_axes = []

# Waffles: each waffle is bottom-anchored ('S') so the bars rise from a
# common baseline; aspect='equal' keeps cells square within the slot.
# Note: set_anchor('S') must be called AFTER Waffle.make_waffle() — the
# pywaffle call resets the axes anchor when it adjusts limits/aspect.
for i, (year, n_solves, n_cases, yield_pct) in enumerate(C_YEARS):
    ax_c = fig.add_subplot(gs_c[0, i])
    ax_c.set_aspect('equal')
    Waffle.make_waffle(
        ax=ax_c,
        columns=C_COLUMNS,
        values=data_c_sorted[year],
        vertical=True,
        colors=colors_c_reversed,
        rounding_rule='ceil',
    )
    ax_c.set_anchor('S')
    panel_c_axes.append(ax_c)

    # Year label below the waffle. Compact to fit narrow year-col slot.
    ax_c_label = fig.add_subplot(gs_c[1, i])
    ax_c_label.axis('off')
    ax_c_label.text(
        0.5, 1.0, f'{year}\n{yield_pct}\n{n_solves}/{n_cases}',
        ha='center', va='top', fontsize=FONTSIZE,
    )
    panel_c_axes.append(ax_c_label)

# Panel C legend (4 main categories, in the natural top-down order).
ax_c_leg = fig.add_subplot(gs_c[:, -1])
ax_c_leg.axis('off')
panel_c_axes.append(ax_c_leg)
patches_c = [
    mpatches.Patch(color=CATEGORY_PRIMARY_COLOR[cat], label=wrapped(cat))
    for cat in CATEGORIES
]
leg_c = ax_c_leg.legend(
    handles=patches_c, loc='upper left', bbox_to_anchor=(0, 1),
    fontsize=FONTSIZE, frameon=False,
    handlelength=1.2, handletextpad=0.6, borderpad=0, labelspacing=0.6,
)
leg_c._legend_box.align = "left"

# ===== Per-panel vertical shifts =====
# Apply AFTER all axes are created so panels B and C move as a unit. The
# panel A legend is already positioned in figure coords above and is not
# affected. Panel C's legend bbox is re-anchored below to follow ax_c_leg.
B_SHIFT = B_SHIFT_MM / FIG_HEIGHT_MM
C_SHIFT = C_SHIFT_MM / FIG_HEIGHT_MM

def shift_axes(ax, dy):
    pos = ax.get_position()
    ax.set_position([pos.x0, pos.y0 + dy, pos.width, pos.height])

shift_axes(ax_b, B_SHIFT)
for ax in panel_c_axes:
    shift_axes(ax, C_SHIFT)

# Re-anchor panel C legend in figure coords to apply the down-nudge AFTER
# the panel C shift (so it tracks ax_c_leg's new position).
ax_c_leg_pos = ax_c_leg.get_position()
leg_c.set_bbox_to_anchor(
    (ax_c_leg_pos.x0, ax_c_leg_pos.y1 + C_LEGEND_Y_NUDGE_MM / FIG_HEIGHT_MM),
    transform=fig.transFigure,
)

# ===== Panel labels (a / b / c / d) =====
# 'a' and 'b' share PANEL_LABEL_X so they're vertically aligned along the
# left margin. 'c' uses B_SHIFT (not C_SHIFT) so it aligns horizontally with
# 'b' regardless of how panel C's content is shifted. 'd' is a placeholder
# for panel D, which is composed manually in Affinity below this label.
def panel_label(spec, letter, x=None, y_shift=0.0):
    pos = spec.get_position(fig)
    if x is None:
        x = max(0.0, pos.x0 - 0.025)
    fig.text(
        x, pos.y1 + PANEL_LABEL_Y_OFFSET + y_shift,
        letter, fontsize=PANEL_LABEL_FONTSIZE, fontweight='bold',
        ha='left', va='top',
    )

panel_label(gs_top[0, :], 'a', x=PANEL_LABEL_X)
panel_label(gs_b[0, 1], 'b', x=PANEL_LABEL_X, y_shift=B_SHIFT)
panel_label(gs_top[1, 1], 'c', y_shift=B_SHIFT)

# 'd' placeholder — adjust PANEL_D_LABEL_Y to where panel D will be added.
fig.text(
    PANEL_LABEL_X, PANEL_D_LABEL_Y,
    'd', fontsize=PANEL_LABEL_FONTSIZE, fontweight='bold',
    ha='left', va='top',
)

fig.savefig('../../Figures/Fig3/Fig3_panels_ABC.pdf')
plt.show()


In [ ]:
! open ../../Figures/Fig3/Fig3_panels_ABC.pdf